In [0]:
df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/ansh/BigMart Sales.csv",inferSchema=True,header=True)
df.display()

# Scanning **Optimization**

### Scanning Optimization is the process of reducing the amount of data Spark reads from storage. 
Since disk I/O is one of the slowest operations, Spark tries to scan as little data as possible.

Goal: Read only the required data (rows and columns) instead of scanning the entire dataset.

In [0]:

from pyspark.sql.functions import col



df_opt=df.filter(col("Outlet_Location_Type")=='Tier 1')
df_opt.display()

In [0]:
df.write.format('csv').mode('overwrite').partitionBy("Outlet_Location_Type").option("path","/Volumes/izwd37dev/wd37db/rawdatta/ansh/scanoptimized").save()

In [0]:
from pyspark.sql.functions import spark_partition_id

# Get number of partitions (serverless-compatible method)
df.select(spark_partition_id().alias("partition")).distinct().count()

In [0]:
df.withColumn("partition_id", spark_partition_id()).display() 

Types of Scanning Optimizations

There are several important scan optimizations:

Column Pruning

Predicate Pushdown

Partition Pruning

Dynamic Partition Pruning

Data Skipping (Delta Lake)

File Pruning

Let's understand each.

In [0]:
#Column Pruning
#reads only required data
df.select("Outlet_Location_Type")

In [0]:
#Predicate Pushdown  --only matching rows were read
df.filter(col("Outlet_Location_Type") == 'Tire 1')

In [0]:
#Partition Pruning  --only matching partitions were read
df.filter(col("Outlet_Location_Type") == 'Tire 1')


In [0]:
#Dynamic Partition Pruning
#it will read only matching rows from another join
'''
SELECT *
FROM sales s
JOIN customer c
ON s.customer_id = c.customer_id
WHERE c.country = 'India' '''

In [0]:
#Data Skipping (Delta Lake)
#skips entire  file completely that does not matching condition and only reads the file that matches the condition
#This avoids unnecessary file reads.
'''
eg : salary > 100000
but in the file max salry is 5000 means , spark will omit the file.'''

In [0]:
#6. File Pruning
'''
Suppose:
employees/
part1.parquet
part2.parquet
part3.parquet

If Spark determines only part2.parquet contains the required data, it skips the other files.'''

Optimization	    What It Does

Column Pruning	 --    Reads only required columns

Predicate Pushdown --	Filters rows at the data source

Partition Pruning --	Reads only required partitions

Dynamic Partition Pruning --	Prunes partitions during joins

Data Skipping	-- Skips files using statistics (Delta Lake)

File Pruning --	Skips unnecessary files

JOIN OPTIMZATION 
SORT MERGE JOIN AND BROADCAST JOIN

In [0]:
emp_data = [
    (101, "Vignesh", 10, 50000),
    (102, "Arun", 20, 60000),
    (103, "Karthik", 30, 70000),
    (104, "Rahul", 40, 80000),
    (105, "Priya", None, 55000)
]

emp_columns = ["emp_id", "emp_name", "dept_id", "salary"]

emp_df = spark.createDataFrame(emp_data, emp_columns)

emp_df.show()

dept_data = [
    (10, "HR"),
    (20, "Finance"),
    (30, "IT"),
    (50, "Marketing")
]

dept_columns = ["dept_id", "dept_name"]

dept_df = spark.createDataFrame(dept_data, dept_columns)

dept_df.show()

In [0]:
from pyspark.sql.functions import broadcast

#SORT MERGE JOIN
df_sortj=emp_df.join(dept_df,'dept_id').display()
#BROADCAST JOIN
df_broadcast=emp_df.join(broadcast(dept_df),'dept_id').display()


# SPARK SQL HINTS

CREATEORREPLACETEMPVIEW -- IT WILL ACCESIBLE ONLY INSIDE THIS NOTEBOOK / SESSION

CREATEORREPLACEGLOBALTEMPVIEW -- IT WILL ACCESIBLE ONLY INSIDE THIS NOTEBOOK / SESSION


In [0]:
emp_df.createOrReplaceTempView("emp")
dept_df.createOrReplaceTempView("dept")
df_sql=spark.sql("select * from emp join dept on emp.dept_id=dept.dept_id").display()

In [0]:
#advice , it will take advice from ours , but it will not execute as gurantee venumna advice eduttukum 
'''
df_sql1=spark.sql("""
                  select * /*+ broadcast(dept) */
                  from emp 
                  join dept 
                  on emp.dept_id=dept.dept_id
                  """).display() '''

In [0]:
#CACHING AND PERSIST
#df.cache()
#df.persist()

# Dynamic Resource Allocation (DRA) :
is a Spark feature that automatically increases the number of executors when there is more work and removes idle executors when demand decreases. This improves cluster utilization, reduces resource waste, and optimizes job performance.


              Driver
                 │
        Dynamic Allocation
                 │
      -------------------------
      │      │      │       │
   Executor Executor Executor Executor
       ↑                     ↓
 Add More              Remove Idle

AQE :

dynamically optimizes the query execution plan during runtime using query statsitics

FEATURES:

1.Dynamically colasece the partitions

2.Optimizing join startegy during runtime

3. optimizing skewness

# Dynamic parttion pruning

In [0]:
# Note: On Serverless compute, AQE and DPP are enabled by default and cannot be configured
# spark.sql.adaptive.enabled - Always enabled (not configurable)
# spark.sql.optimizer.dynamicPartitionPruning.enabled - Managed by platform (not configurable)
#spark.conf.set("spark.sql.autoBroadcastJoinThreshold",5*1024*1024)

# What is a Broadcast Variable?

A Broadcast Variable is a read-only variable that is sent once from the Driver to all Executors, allowing tasks to reuse the same data without sending it repeatedly.

Think of it as giving every executor its own copy of a small reference dataset.

In [0]:
#df
dfb=spark.createDataFrame([("1",),("2",)],["product_id"])
#lookup dictinary small

product_dict={"1":"iphone","2":"samsung"}
'''
#broadcasting the varibale
broad_vr=spark.sparkContext.broadcast(product_dict)
broad_vr.value
broad_vr
'''


In [0]:
tax_rate = spark.sparkContext.broadcast(0.18)
tax_rate.value

In [0]:
def mymap(x):
    return broad.vr.value.get(x)
mymap_udf=udf(mymap)

In [0]:
df_with_names=emp_df.withColumn("product_name",mymap_udf("product_id"))
df_with_names.show()

SALZTING

In [0]:
data=[("A",100),("A",200),("A",300),("B",100),("B",200)]
df=spark.createDataFrame(data,["name","age"])
df.show()
from pyspark.sql.functions import *


In [0]:
#ADDING SALT COLUMN
df.withColumn("salt_column",floor(rand()*3))

In [0]:
#creating concat 
df.withColumn("concat_col",concat(col("name"),lit("_"),col("salt_column"))).display()

In [0]:
#applying group by on the new column
df.withColumn("salt_column",floor(rand()*3)).withColumn("concat_col",concat(col("name"),lit("_"),col("salt_column"))).groupBy("concat_col").agg(sum("age")).display()

Delta Lake optimization 

is the process of improving query performance and storage efficiency using techniques such as file compaction (OPTIMIZE), Z-Ordering, partitioning, data skipping, predicate pushdown, column pruning, caching, and VACUUM. These optimizations reduce the amount of data scanned, minimize small-file overhead, and speed up query executi

In [0]:
%sql
create schema ansh

In [0]:
%sql
create table ansh.anshtbl
(id int,
salry int
)
using delta
location '"/Volumes/izwd37dev/wd37db/rawdatta/ansh"'


In [0]:
%sql
OPTIMIZE delta.`/Volumes/izwd37dev/wd37db/rawdatta/ansh` ZORDER BY (id)
    
DESCRIBE DETAIL delta.`/Volumes/izwd37dev/wd37db/rawdatta/ansh`
    
select * from ansh.anshtbl